In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [4]:
# Load data
df = pd.read_csv('../data/processed/cleaned_data.csv')
complaints = pd.read_csv('../data/references/youtube_premium_complain.csv')
churn_ts = pd.read_csv('../data/references/youtube_churn.csv')
family_plan = pd.read_csv('../data/references/family_plan.csv')
prices = pd.read_csv('../data/references/youtube_price.csv')

In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5534 entries, 0 to 5533
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5534 non-null   object 
 1   gender            5534 non-null   object 
 2   SeniorCitizen     5534 non-null   int64  
 3   Partner           5534 non-null   object 
 4   Dependents        5534 non-null   object 
 5   tenure            5534 non-null   float64
 6   PhoneService      5534 non-null   object 
 7   MultipleLines     5534 non-null   object 
 8   InternetService   5534 non-null   object 
 9   OnlineSecurity    5534 non-null   object 
 10  OnlineBackup      5534 non-null   object 
 11  DeviceProtection  5534 non-null   object 
 12  TechSupport       5534 non-null   object 
 13  StreamingTV       5534 non-null   object 
 14  StreamingMovies   5534 non-null   object 
 15  Contract          5534 non-null   object 
 16  PaperlessBilling  5534 non-null   object 


In [6]:
df['calculated_tenure'] = (df['TotalCharges'] / df['MonthlyCharges']).round().astype(int)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,calculated_tenure
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,33
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,2
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,44
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2


#### 4 Scenarios between Aug 2025 and Feb 2026 on NO Churn
A. In February 2026, they either:
1. Family→Individual
- Had family plan in Aug
  - Switched to individual by Feb
  - Interpretation: Price-sensitive, left family plan
  - Churn status: Lost to competitor or downgraded
2. Family → Family
- Had family plan in Aug
  - Maintained family plan in Feb
  - Interpretation: Loyal, accepted pricing or valued service
  - Churn status: Retained

B. 
3. Individual → Individual
4. Individual → Family

[For Stable Price Group]

"When tenure_diff is near zero, it indicates Price Stability. Since their monthly bill hasn't changed since 2025, any plan transitions (like Family to Individual) are likely due to lifestyle changes, not financial reasons."

[For Price Increase Group]

"A positive tenure_diff means the customer is currently paying more than their historical average. We categorize them as 'High-Risk' if their plan remained the same, as they are experiencing price pressure. However, for those who switched from Individual to Family, this increase is a natural result of a service upgrade."

#### IMPORTANT - Future Phase (Pricing Elasticity)

**NOTE: Charge Change Analysis (Later Phase)**

1. Current Phase Goal:

- Identify survivors by subscription change type
- Measure loyalty/resilience
- Understand "who stayed despite policy change"

2. Future Phase Goal:

- Use charge_change to predict price elasticity
- Calculate revenue optimal price point

3. Key Combinations (Phase 3 - Pricing Model):
- ScenarioPatternMeaningFamily→Individual + charge_change > 0Price-driven exitLeft because cost increased
- Family→Family + charge_change ≈ 0Loyal + acceptable pricingStayed despite/with similar price
- Individual→Individual + charge_change > 0Loyal despite increase
- Stayed even though price roseIndividual→Family + charge_change < 0Attracted by discountSwitched due to lower price

##### ACTION: Save this logic for Phase 3 (Pricing Model)

In [7]:
# Filtering Subscribers until Aug 2025

# Set reference date
reference_date = pd.to_datetime('2026-02-01')

# Calculate signup date: Current Date - Past Date
#DateOffset:Focuses on relative calendar dates --> calculate one value at once >> need "apply"
df['signup_date'] = df['calculated_tenure'].apply(
                    lambda x: reference_date -  pd.DateOffset(months=int(x))
                    )

# Filtering (to prevent 0 division due to tenure = 1)
baseline = df[(df['calculated_tenure'] > 5) &
              (df['calculated_tenure'] != 7)
              ].copy()


# Using Filtered baseline, Calculate reverse-tracking indicators

# 1. Calculate past average monthly charges (excluding Feb)
baseline['past_monthly_charges'] = (
    (baseline['TotalCharges'] - baseline['MonthlyCharges']) / 
    (baseline['calculated_tenure'] - 1)
)

# 2. Calculate - charge consistency check 
# Calculate Consistency Ratio (How many months of 'current price' fit into 'total charges')
baseline['consistency_check'] = baseline['TotalCharges'] / baseline['MonthlyCharges']

# 3. Calculate tenure difference
baseline['tenure_diff'] = baseline['calculated_tenure'] - baseline['consistency_check']

# 4. 4 scenarios using charge_change

baseline['charge_change'] = baseline['MonthlyCharges'] - baseline['past_monthly_charges']

# Infer Aug status using charge_change
baseline['family_aug'] = baseline['charge_change'].apply(
    lambda x: False if x < -6 else True
)

# Feb status
baseline['family_feb'] = (baseline['Partner'] == 'Yes') | (baseline['Dependents'] == 'Yes') 

# Classify 4 scenarios
baseline['scenario'] = baseline.apply(
    lambda row: 'Family --> Family' if (row['family_aug'] and row['family_feb'])
           else 'Family --> Individual' if (row['family_aug'] and not row['family_feb']) 
           else 'Individual --> Individual' if (not row['family_aug'] and not row['family_feb'])
           else 'Individual --> Family',
    axis=1
)

print("\n*** 4 Scenarios ***")
print(baseline['scenario'].value_counts())


*** 4 Scenarios ***
scenario
Family --> Family            2633
Family --> Individual        1745
Individual --> Individual       6
Individual --> Family           2
Name: count, dtype: int64


In [8]:
# Verification
# Step 1: Total count check
print("=== Validation ===")
total_scenarios = baseline['scenario'].value_counts().sum()
print(f"Total from scenarios: {total_scenarios}")
print(f"Total baseline: {len(baseline)}")
print(f"Match: {total_scenarios == len(baseline)}")

# Step 2: Check Feb status
print("\n=== Feb Status (should match scenarios) ===")
is_family_feb = (baseline['Partner'] == 'Yes') | (baseline['Dependents'] == 'Yes')
print(f"Feb Family (Partner/Dependents = Yes): {is_family_feb.sum()}")
print(f"Feb Individual (both No): {(~is_family_feb).sum()}")

# Step 3: Verify scenario mapping
print("\n=== Scenario Breakdown ===")
family_feb = baseline[is_family_feb]['scenario'].value_counts()
individual_feb = baseline[~is_family_feb]['scenario'].value_counts()

print("Feb Family scenarios:")
print(family_feb)
print("\nFeb Individual scenarios:")
print(individual_feb)

# Step 4: Cross-check
print("\n=== Cross-Check ===")
scenarios = baseline['scenario'].value_counts()
family_feb_total = scenarios.get('Family → Family', 0) + scenarios.get('Individual → Family', 0)
individual_feb_total = scenarios.get('Family → Individual', 0) + scenarios.get('Individual → Individual', 0)

print(f"Family → Family + Individual → Family = {family_feb_total}")
print(f"Family → Individual + Individual → Individual = {individual_feb_total}")
print(f"Should match Feb Family: {is_family_feb.sum()}")
print(f"Should match Feb Individual: {(~is_family_feb).sum()}")

=== Validation ===
Total from scenarios: 4386
Total baseline: 4386
Match: True

=== Feb Status (should match scenarios) ===
Feb Family (Partner/Dependents = Yes): 2635
Feb Individual (both No): 1751

=== Scenario Breakdown ===
Feb Family scenarios:
scenario
Family --> Family        2633
Individual --> Family       2
Name: count, dtype: int64

Feb Individual scenarios:
scenario
Family --> Individual        1745
Individual --> Individual       6
Name: count, dtype: int64

=== Cross-Check ===
Family → Family + Individual → Family = 0
Family → Individual + Individual → Individual = 0
Should match Feb Family: 2635
Should match Feb Individual: 1751


In [9]:


# Split baseline into Survivors and Churners - follow the same (tenure > 5) filter logic
survivors = baseline[baseline['Churn'] == 'No'].copy()
churners = baseline[baseline['Churn'] == 'Yes'].copy()

# Define Family vs Individual in the Churn group
family_churners = churners[
                    (churners['Partner'] == 'Yes') | (churners['Dependents'] == 'Yes')
]

individual_churners = churners[
                    (churners['Partner'] == 'No') | (churners['Dependents'] == 'No')
]

# Define Family vs Individual in the Survivors group
family_survivors = survivors[
                    (survivors['Partner'] == 'Yes') | (survivors['Dependents'] == 'Yes')
]

individual_survivors = survivors[
                    (survivors['Partner'] == 'No') | (survivors['Dependents'] == 'No')
]

# Calculate Final Churn Rate
family_churn_rate = (len(family_churners)) / (len(family_survivors) + len(family_churners)) * 100
indiv_churn_rate = (len(individual_churners)) / (len(individual_survivors) + len(individual_churners)) * 100


print(f"*** Churn Rate: Baseline Subset ***")
print(f"Family Churn Rate: {family_churn_rate:.2f}")
print(f"Individual Churn Rate: {indiv_churn_rate:.2f}")


*** Churn Rate: Baseline Subset ***
Family Churn Rate: 16.32
Individual Churn Rate: 22.90
